### Initial ML Model for Delsys Data Input (LDA Model)
Consists of:
- Filtering
- Windowing
- Feature Extraction
- Training LDA
- Evaluating accuracy
- Works for ADLs like “water-bottle lift” and “zipper”

First setting up virtual environment:
- py -m venv venv
- venv\Scripts\activate

Then setting up dependencies:
- pip install numpy scipy scikit-learn matplotlib seaborn

##### Imports

In [78]:
import pandas as pd
import numpy as np
import re
from scipy.signal import butter, filtfilt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import classification_report, confusion_matrix

##### Functions

In [79]:
import pandas as pd
import numpy as np

def load_emg(csv_path):
    """
    Load EMG CSV from Trigno Discover.
    Returns:
        emg_data: dict {Sensor1: (n_samples, 1), Sensor2: (n_samples, 1), ...}
        time_data: dict {Sensor1: (n_samples, 1), ...}  # optional
    """
    # --- Read sensor row (row 4) ---
    sensor_row = pd.read_csv(csv_path, header=None, skiprows=3, nrows=1, skipinitialspace=True)
    sensor_row = sensor_row.ffill(axis=1).iloc[0].tolist()

    # --- Read measurement row (row 6) ---
    meas_row = pd.read_csv(csv_path, header=None, skiprows=5, nrows=1, skipinitialspace=True)
    meas_row = meas_row.iloc[0].tolist()

    # --- Make unique sensor names ---
    combined_cols = []
    sensor_counter = {}
    for sensor, meas in zip(sensor_row, meas_row):
        sensor = str(sensor) if pd.notna(sensor) else "Unknown"
        meas = str(meas) if pd.notna(meas) else "Unknown"

        # Count duplicates
        sensor_counter[sensor] = sensor_counter.get(sensor, 0) + 1
        unique_sensor = f"{sensor}_{sensor_counter[sensor]}"  # e.g., Galileo Sensor 1_1
        combined_cols.append(f"{unique_sensor}_{meas}")

    # --- Load data starting row 9 ---
    df = pd.read_csv(csv_path, header=None, skiprows=8, skipinitialspace=True, low_memory=False)

    # Pad column names if needed
    n_cols = df.shape[1]
    if len(combined_cols) < n_cols:
        last_sensor = combined_cols[-1].split("_")[0]
        for i in range(n_cols - len(combined_cols)):
            combined_cols.append(f"{last_sensor}_extra{i}")
    elif len(combined_cols) > n_cols:
        combined_cols = combined_cols[:n_cols]

    df.columns = combined_cols

    # Convert all to numeric safely
    df = df.apply(lambda x: pd.to_numeric(x.astype(str).str.strip(), errors='coerce')).fillna(0)

    # Separate EMG and time columns
    emg_data = {}
    time_data = {}
    for col in df.columns:
        if "(mV)" in col:
            sensor_name = col.split("_")[0]  # unique sensor
            emg_data[sensor_name] = df[[col]].to_numpy()  # shape (n_samples,1)
        elif "Time Series" in col:
            sensor_name = col.split("_")[0]
            time_data[sensor_name] = df[[col]].to_numpy()

    return emg_data, time_data


In [80]:
#def bandpass_filter(data, lowcut=20, highcut=450, fs=963, order=4):
#    """
#    Apply zero-phase 4th-order Butterworth bandpass filter to EMG data.
#    """
#    nyq = 0.5 * fs
#    low = lowcut / nyq
#    high = highcut / nyq
#    b, a = butter(order, [low, high], btype='band')
#    filtered = filtfilt(b, a, data, axis=0)
#    return filtered

In [81]:
def extract_features(emg_window):
    """
    Extract features from a 1D EMG array (win_size,).
    Returns a 1D array of features: [MAV, RMS, WL]
    """
    emg_window = np.ravel(emg_window)  # ensure 1D
    mav = np.mean(np.abs(emg_window))
    rms = np.sqrt(np.mean(emg_window**2))
    wl = np.sum(np.abs(np.diff(emg_window)))
    return np.array([mav, rms, wl])

In [82]:
def window_emg(emg_data, fs=963, window_sec=0.2, overlap_sec=0.1):
    """
    Slice EMG data (n_samples x n_sensors) into overlapping windows.
    Returns features concatenated across sensors.
    """
    win_size = int(window_sec * fs)
    step = int((window_sec - overlap_sec) * fs)

    # Ensure 2D array
    if emg_data.ndim == 1:
        emg_data = emg_data[:, np.newaxis]

    n_samples, n_sensors = emg_data.shape
    features = []

    for start in range(0, n_samples - win_size, step):
        window = emg_data[start:start + win_size, :]  # (win_size, n_sensors)
        window_feats = []
        for ch in range(n_sensors):
            ch_feat = extract_features(window[:, ch])
            window_feats.extend(ch_feat)
        features.append(window_feats)

    return np.array(features)

In [83]:
def process_trial(csv_path, label, fs=963):
    """
    Load EMG, apply filtering, windowing, and return features + labels.
    """
    emg_data, _ = load_emg(csv_path)
    X = window_emg(emg_data, fs=fs)
    y = np.full(X.shape[0], label)
    return X, y

In [84]:
def load_all_trials(class_trials, fs=963, window_sec=0.2, overlap_sec=0.1):
    """
    Load multiple trials and generate X, y arrays.
    class_trials: dict {label: [file1, file2, ...]}
    Returns:
        X: (n_windows_total, n_features)
        y: (n_windows_total,)
    """
    X_all = []
    y_all = []

    for label, files in class_trials.items():
        for file in files:
            emg_dict, _ = load_emg(file)
            # Concatenate all sensors into (n_samples, n_sensors)
            emg_list = [emg_dict[s] for s in sorted(emg_dict.keys())]
            emg_data = np.column_stack(emg_list)

            # Window and extract features
            X_trial = window_emg(emg_data, fs=fs, window_sec=window_sec, overlap_sec=overlap_sec)
            y_trial = np.full(X_trial.shape[0], label)

            X_all.append(X_trial)
            y_all.append(y_trial)

    X_all = np.vstack(X_all)
    y_all = np.concatenate(y_all)
    return X_all, y_all

In [85]:
def prepare_data(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test

In [86]:
def train_evaluate(X_train, X_test, y_train, y_test):
    clf = LinearDiscriminantAnalysis()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    return clf

##### Load and process ADL data

X_lift, y_lift = process_trial("20251113-Data/Lifting.csv", label=0)
X_zip, y_zip = process_trial("20251113-Data/Zipping.csv", label=1)
X_pinch, y_pinch = process_trial("20251113-Data/Pinching.csv", label=2)

X = np.vstack([X_lift, X_zip, X_pinch])
y = np.concatenate([y_lift, y_zip, y_pinch])

In [87]:
import os
import glob

def get_trials_from_folder(data_dir):
    """
    Scan the folder and parse filenames to automatically assign class labels
    based on the first number in the filename.
    Returns a dict: {class_label: [file_paths]}
    """
    trial_files = glob.glob(os.path.join(data_dir, "*.csv"))
    class_trials = {}

    for f in trial_files:
        basename = os.path.basename(f)
        # Expecting filenames like "0.1_name_date.csv", "1.2_name_date.csv", etc.
        try:
            class_label = int(basename.split(".")[0])  # take first number as class
        except ValueError:
            print(f"Skipping file with unexpected name format: {basename}")
            continue

        if class_label not in class_trials:
            class_trials[class_label] = []
        class_trials[class_label].append(f)

    return class_trials

# Parse files and assign class labels automatically
class_trials = get_trials_from_folder("20251201-Data")
print("Found class trials:", {k: len(v) for k, v in class_trials.items()})

# Load and process all trials
X, y = load_all_trials(class_trials)

# Split and scale
X_train, X_test, y_train, y_test = prepare_data(X, y)

# Train and evaluate LDA model
clf = train_evaluate(X_train, X_test, y_train, y_test)

Found class trials: {0: 6, 1: 8, 2: 8, 3: 4}
Classification Report:
               precision    recall  f1-score   support

           0       0.68      0.94      0.79       528
           1       0.65      0.67      0.66       696
           2       0.68      0.56      0.61       860
           3       0.55      0.45      0.50       457

    accuracy                           0.65      2541
   macro avg       0.64      0.66      0.64      2541
weighted avg       0.65      0.65      0.64      2541

Confusion Matrix:
 [[496   7   3  22]
 [ 64 468 123  41]
 [108 162 481 109]
 [ 66  83 101 207]]
